# Chapter 13: Ethics, Fairness, and Data

**Foundations of Data Science**

## In this lesson

1. A model can be accurate overall and wrong for a group: why averages hide it
2. Where unfairness enters: the label, the features, the sample, the cutoff
3. Fairness metrics, and why you can't have all of them
4. Auditing the chapter 8 screening model by race/ethnicity and sex
5. Bootstrapped intervals and tests on group differences
6. What to do when the audit fails
7. Beyond the model: consent, privacy, and who bears the cost of errors
8. A checklist

## The one idea

**Every metric from chapter 8 can be computed per group, and the differences between groups are a property of the model you're responsible for.** An AUC of 0.80 is an average over everyone. If it's 0.85 for one group and 0.70 for another, the model is a different product for different people, and the people it works worst for are often the ones with the least power to object.

The clinic question, final version: **the chapter 8 screening rule flags about 43% of adults and catches 80% of cases. Is that true for every group it will be applied to? If not, who is being missed, and what should change?**

## Setup

<!-- nb-links -->
[Download this notebook](chapter_13_fairness.ipynb?download=1){download="chapter_13_fairness.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/chapter_13/chapter_13_fairness.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Refit the chapter 8 model

Same pipeline, same split, same chosen hyperparameters, same 0.16 cutoff. The audit is on the test set, which is the only place it's honest.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from model_tuner import Model

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[nhanes["age"] >= 20].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["race_eth"] = adults["race_eth"].map({1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other/multi"})
num = ["age", "bmi", "waist_cm", "sbp", "dbp", "income_ratio"]
cat = ["sex", "race_eth"]
X, y = adults[num + cat], adults["diabetes"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat),
])

def make_model(name, estimator, grid):
    return Model(name=name, estimator_name=name, estimator=estimator, model_type="classification",
                 pipeline_steps=[("pre", preprocess)], grid=grid, scoring=["roc_auc"],
                 stratify_y=True, train_size=0.6, validation_size=0.2, test_size=0.2, random_state=42, display=False)

lr = make_model("lr", LogisticRegression(max_iter=2000), [{"lr__C": [10]}])
lr.grid_search_param_tuning(X, y)
X_train, y_train = lr.get_train_data(X, y)
X_test, y_test = lr.get_test_data(X, y)
lr.fit(X_train, y_train)

p_lr = lr.predict_proba(X_test)[:, 1]
THRESHOLD = 0.16                                   # the 80%-sensitivity cutoff chosen in chapter 8
pred_lr = (p_lr >= THRESHOLD).astype(int)
print("test rows:", len(y_test), "  cases:", int(y_test.sum()), "  flagged:", int(pred_lr.sum()))

## 2. Where unfairness enters

A model can disadvantage a group without anyone intending it. Four doors, with this dataset's version of each:

| Door | Mechanism | Here |
|---|---|---|
| **The label** | the outcome recorded isn't the outcome you care about | `diabetes` is *self-reported doctor diagnosis*. Groups with less healthcare access have more undiagnosed diabetes, so the label undercounts them, and a model trained on it learns to undercount them too |
| **The features** | a feature carries different information for different groups | BMI understates metabolic risk for Asian adults (chapter 7's odds ratios showed it); blood pressure cuffs misread on large arms (chapter 2) |
| **The sample** | some groups are too small to learn from | `Other/multi` has ~270 adults; the model's behavior there is barely constrained by data |
| **The cutoff** | one threshold, chosen on the pooled data, lands differently on groups with different score distributions | a cutoff that gives 80% sensitivity overall can give 90% in one group and 60% in another |

The famous case: Obermeyer et al. (2019) found a widely used care-management algorithm used *healthcare cost* as its label for *healthcare need*. Black patients generated lower costs at the same level of illness (less access, less spending on them), so the model rated them healthier than they were. Door one. Nothing in the features mentioned race; the label did the damage.

## 3. Fairness metrics, and why you can't have them all

Take any confusion-matrix rate from chapter 8 and ask whether it's equal across groups. The common ones, each a different claim about what "fair" means:

| Criterion | Equal across groups | Whose concern |
|---|---|---|
| **Demographic parity** | share flagged (predicted prevalence) | equal burden of screening |
| **Equal opportunity** | sensitivity (true positive rate) | equal chance of being caught if you have it |
| **Equalized odds** | sensitivity *and* false positive rate | the above, plus equal burden of false alarms |
| **Predictive parity** | PPV (precision) | a flag means the same thing for everyone |
| **Calibration** | predicted probability matches observed rate, within group | a stated 20% is 20% for everyone |

A known result (Chouldechova 2017; Kleinberg et al. 2016): when base rates differ between groups, **you cannot have calibration, equal sensitivity, and equal false positive rate at once**, except in the trivial case of a perfect model. Something has to give, and which thing gives is a decision about values, not a modeling choice. For screening, the usual stance is: equalize *sensitivity* (every group's cases get caught at the same rate), and accept that the false-positive burden will differ. Say which you chose and why.

## 4. The audit

`equiboots` takes the true labels, predictions, probabilities, and the group columns, and computes every chapter 8 metric per group, plus **disparity ratios** (each group's metric divided by a reference group's).

In [ ]:
import equiboots as eb

fairness_df = X_test[["race_eth", "sex"]].reset_index(drop=True)
audit = eb.EquiBoots(
    y_true=y_test.values, y_pred=pred_lr, y_prob=p_lr,
    fairness_df=fairness_df, fairness_vars=["race_eth", "sex"],
    reference_groups=["White", "female"], task="binary_classification", bootstrap_flag=False,
)
audit.grouper(groupings_vars=["race_eth", "sex"])

by_race = audit.get_metrics(audit.slicer("race_eth"))
rows = ["Prevalence", "Predicted Prevalence", "Recall", "FP Rate", "Precision", "ROC AUC", "Brier Score", "TP", "FN"]
eb.metrics_table(by_race).loc[rows].round(3)

In [ ]:
# Disparity ratios: each group's metric divided by the White group's. 1.0 is parity; the usual "four-fifths" screen flags anything outside 0.8 to 1.25.
disp_race = audit.calculate_disparities(by_race, "race_eth")
disp_tbl = pd.DataFrame(disp_race).loc[["Recall_Ratio", "FP_Rate_Ratio", "Precision_Ratio", "Predicted_Prevalence_Ratio", "ROC_AUC_Ratio"]].round(2)
disp_tbl

In [ ]:
# Same by sex
by_sex = audit.get_metrics(audit.slicer("sex"))
print(eb.metrics_table(by_sex).loc[rows].round(3).to_string())
print()
print(pd.DataFrame(audit.calculate_disparities(by_sex, "sex")).loc[["Recall_Ratio", "FP_Rate_Ratio", "Precision_Ratio", "Predicted_Prevalence_Ratio"]].round(2).to_string())

In [ ]:
# ROC and calibration curves per group. The ROC curves say whether the model ranks equally well within each group;
# the calibration curves say whether a given probability means the same thing in each group.
sliced = audit.slicer("race_eth")
eb.eq_plot_group_curves(sliced, curve_type="roc", title="ROC by race/ethnicity", figsize=(7, 5.5))
eb.eq_plot_group_curves(sliced, curve_type="calibration", n_bins=6, title="Calibration by race/ethnicity", figsize=(7, 5.5))

Read the tables with the chapter 8 cutoff in mind: it was chosen to give 80% sensitivity *overall*.

- **Sensitivity** (equal opportunity): White, Black, Asian, and Mexican American adults all land at 0.80 to 0.89. Other Hispanic adults are at 0.62 (13 of 21 cases caught) and Other/multi at 0.60 (6 of 10). By sex: men 0.84, women 0.72, and that one is on 94 and 81 cases, so it's not a small-sample artifact. At this cutoff the rule misses women and two smaller ethnic groups at a higher rate than everyone else.
- **False-positive rate and predicted prevalence**: the share flagged runs from 29% (Other Hispanic) to 52% (Mexican American), tracking base rates and score distributions. Groups with higher prevalence get flagged more, which is what a calibrated model does and which demographic parity would call unfair. This is the impossibility result on the page.
- **AUC** is 0.79 to 0.84 everywhere: the model ranks about equally well within each group. Ranking isn't the problem; the single cutoff is.
- **Calibration**: the curves bend differently by group where the test-set bins are thin. Small groups give noisy calibration curves, which is itself a finding about sample size, not about the model.

## 5. Intervals and tests on the differences

The `Other/multi` group has about 50 people in the test set and a handful of cases. Its sensitivity could be 0.6 or 0.9 with a different draw. Before calling a disparity real, bootstrap the test set within each group and test whether each group's metric differs from the reference.

In [ ]:
audit_boot = eb.EquiBoots(
    y_true=y_test.values, y_pred=pred_lr, y_prob=p_lr,
    fairness_df=fairness_df, fairness_vars=["race_eth"], reference_groups=["White"],
    task="binary_classification", bootstrap_flag=True, num_bootstraps=300, boot_sample_size=len(y_test), balanced=False,
)
audit_boot.grouper(groupings_vars=["race_eth"])
boot_metrics = audit_boot.get_metrics(audit_boot.slicer("race_eth"))

# Per-bootstrap differences from the reference group, then a test on each difference's distribution
boot_diffs = audit_boot.calculate_differences(boot_metrics, "race_eth")
tests = audit_boot.analyze_statistical_significance(
    boot_metrics, "race_eth",
    test_config={"test_type": "bootstrap_test", "alpha": 0.05, "adjust_method": "bonferroni",
                 "confidence_level": 0.95, "tail_type": "two_tailed",
                 "metrics": ["Recall_diff", "FP_Rate_diff", "Precision_diff", "ROC_AUC_diff"]},
    differences=boot_diffs,
)
sig = pd.DataFrame({g: {m: r.is_significant for m, r in res.items()} for g, res in tests.items()})
print("significantly different from White (Bonferroni-adjusted):")
sig

In [ ]:
# Forest plot: bootstrapped sensitivity per group with intervals, reference group marked
eb.eq_plot_bootstrap_forest(boot_metrics, metric="Recall", reference_group="White", include_reference_group=True,
                            title="Sensitivity by race/ethnicity at the 0.16 cutoff, bootstrap 95% intervals", figsize=(7, 4))

The intervals do the work the point estimates can't. After a Bonferroni correction for testing five groups on four metrics, the only difference from White that survives is the Asian group's lower false-positive rate. The sensitivity gaps for Other Hispanic (0.62) and Other/multi (0.60) do not: with 21 and 10 cases, their intervals are wide enough to include 0.80. A disparity that disappears inside its interval isn't evidence of nothing; it's evidence that the test set is too small to know, which is a reason to get more data from that group (or pool test sets across NHANES cycles) before shipping, not a reason to ship. The sex gap, by contrast, is on large groups and would be the first thing to fix.

## 6. What to do when the audit fails

In rough order of preference:

1. **Fix the label or the data.** If the label undercounts a group (door one), no amount of modeling fixes it; find a better outcome definition or collect it. For diabetes, lab-measured HbA1c (NHANES has it, `GHB_J`) instead of self-report is the obvious move; rerunning this audit with that label is a good exercise.
2. **Group-specific cutoffs.** Choose a threshold per group that equalizes the metric you care about (sensitivity, usually); `equiboots.find_group_thresholds` matches each group's operating point to a reference group's, or you can target a metric directly. This is legal and standard in clinical screening (different reference ranges by sex are the same idea) and contested elsewhere; know your domain's rules.
3. **Add or fix features.** If BMI means something different by group, the model needs the interaction (chapter 6) or a better feature (waist-to-height).
4. **Constrain the model.** Fairness-aware training (reweighting, constraints) trades overall accuracy for group parity. Last resort, because it leaves the broken label in place.
5. **Don't deploy for that group**, and say so, rather than deploy something you know fails them.

In [ ]:
# Option 2, demonstrated: a cutoff per race/ethnicity group that targets 80% sensitivity in each.
# (equiboots.find_group_thresholds does a related thing: it matches each group's operating point to a reference group's.
#  Here the target is explicit, so the search is written out.)
from sklearn.metrics import recall_score

def threshold_for_recall(y, p, target=0.80):
    # highest cutoff whose sensitivity is still at or above the target
    for t in np.arange(0.5, 0.0, -0.005):
        if recall_score(y, (p >= t).astype(int)) >= target:
            return round(float(t), 3)
    return 0.0

groups = fairness_df["race_eth"].values
group_thresholds = {g: threshold_for_recall(y_test.values[groups == g], p_lr[groups == g]) for g in np.unique(groups)}
print(group_thresholds)

pred_group = eb.grouped_threshold_predict(y_prob=p_lr, group_labels=groups, group_thresholds=group_thresholds)
audit_gt = eb.EquiBoots(y_true=y_test.values, y_pred=pred_group, y_prob=p_lr, fairness_df=fairness_df, fairness_vars=["race_eth"],
                        reference_groups=["White"], task="binary_classification", bootstrap_flag=False)
audit_gt.grouper(groupings_vars=["race_eth"])
eb.metrics_table(audit_gt.get_metrics(audit_gt.slicer("race_eth"))).loc[["Recall", "FP Rate", "Predicted Prevalence"]].round(3)

Sensitivity is now near 0.8 in every group, and the cost shows up where the impossibility result said it would: false-positive rates and share flagged now differ *more* between groups than before. Equalizing one thing un-equalizes another. The clinic has to pick, and the data scientist's job is to make the trade visible, in a table like this one, before anyone picks.

## 7. Beyond the model

Fairness metrics are the measurable part. Three things that aren't:

- **Consent and secondary use.** NHANES participants consented to a public-health survey. A hospital's patients consented to treatment. Training a model on their records is a secondary use; the rules for it (IRB, HIPAA, GDPR) exist because consent to one thing is not consent to everything. The chapter 12 rule (PHI never leaves the building) is the operational version.
- **Who bears the cost of errors.** A false negative costs the patient years of untreated disease. A false positive costs the system a test. The person who bears the cost of the model being wrong is rarely the person who chose the cutoff. Write down who pays for each kind of error before choosing a metric.
- **Feedback.** A model that flags fewer people in a group produces fewer diagnoses in that group, which produces training data with fewer cases in that group, which trains the next model to flag fewer. Audits have to repeat, on new data, after deployment.

## 8. A checklist for every model you ship

1. What is the label, and for whom is it a worse proxy for the real outcome?
2. Which groups are in the data, how big is each in the *test* set, and which are too small to evaluate?
3. Per-group: sensitivity, FPR, PPV, AUC, calibration, with bootstrap intervals.
4. Which fairness criterion did you choose, and what did you give up?
5. Who bears the cost of each error type, and did they have a say?
6. When is the next audit, and on what data?

## Looking ahead

- **Project**: final notebooks and talks next session. Every project's evaluation section needs a per-group table and one sentence on who bears the cost of its errors.
- **Reading**: Obermeyer et al. (2019) in full; Barocas, Hardt & Narayanan chapter 3 if you didn't already.